In [1]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import requests
import os
from langchain_community.tools import DuckDuckGoSearchRun
from langsmith import Client
from langchain_classic.agents import create_react_agent, AgentExecutor

load_dotenv()

/Users/noamankhan/Desktop/Learning/GenAI/LangChain/LangChainPractice/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/7n/9ddnlj_93hd6z10bnfbvxtdm0000gn/T/ipykernel_6258/4066974714.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


True

# Tool Creation

### Weather Tool

In [2]:
@tool
def get_weather(location: str) -> dict:
    """
    Get the current weather and a 14-day forecast for a location.

    Args:
        location: City, country, or coordinates.

    Returns:
        Dictionary containing current weather and 14-day forecast.
    """

    api_key = os.getenv("WEATHER_API_KEY")
    if not api_key:
        raise ValueError("WEATHER_API_KEY is not configured.")

    url = "https://api.weatherapi.com/v1/forecast.json"

    params = {
        "key": api_key,
        "q": location,
        "days": 14,
        "aqi": "no",
        "alerts": "no",
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()

        data = response.json()

        if "error" in data:
            return {"error": data["error"]["message"]}

        current = data["current"]

        forecast = []

        for day in data["forecast"]["forecastday"]:
            forecast.append({
                "date": day["date"],
                "condition": day["day"]["condition"]["text"],
                "max_temp_c": day["day"]["maxtemp_c"],
                "min_temp_c": day["day"]["mintemp_c"],
                "avg_temp_c": day["day"]["avgtemp_c"],
                "chance_of_rain": day["day"]["daily_chance_of_rain"],
                "max_wind_kph": day["day"]["maxwind_kph"],
                "uv": day["day"]["uv"]
            })

        return {
            "location": {
                "city": data["location"]["name"],
                "country": data["location"]["country"],
                "local_time": data["location"]["localtime"],
            },
            "current": {
                "temperature_c": current["temp_c"],
                "feels_like_c": current["feelslike_c"],
                "condition": current["condition"]["text"],
                "humidity": current["humidity"],
                "wind_kph": current["wind_kph"],
                "uv": current["uv"],
                "is_day": bool(current["is_day"])
            },
            "forecast": forecast,
        }

    except requests.exceptions.RequestException as e:
        return {"error": str(e)}

### Currency Conversion Tool

In [3]:
class ConversionInput(BaseModel):
    base_curr: str = Field(
        description="The base currency code (e.g., USD, INR)."
    )
    target_curr: str = Field(
        description="The target currency code (e.g., EUR, JPY)."
    )

In [4]:
@tool
def get_conversion_rate(query: str) -> dict:
    """
    Fetch and return the real-time exchange rate between two currencies.

    Get the conversion rate.

    Input format:
    INR EUR

    Example:
    INR EUR
    """

    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise ValueError("EXCHANGE_RATE_API_KEY is not configured.")

    base_curr, target_curr = query.upper().split()

    url = (
        f"https://v6.exchangerate-api.com/v6/"
        f"{api_key}/pair/{base_curr}/{target_curr}"
    )

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()

    if data["result"] != "success":
        raise ValueError(data.get("error-type", "Currency conversion failed."))

    return {
    "base_currency": base_curr,
    "target_currency": target_curr,
    "conversion_rate": data["conversion_rate"]
}

In [5]:
get_conversion_rate.invoke({"query":"EUR INR"})

{'base_currency': 'EUR', 'target_currency': 'INR', 'conversion_rate': 107.9661}

### Web Search Tool

In [6]:
search_tool = DuckDuckGoSearchRun()

# Model Creation

In [19]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite") 

# Pull ReAct Prompt from Langchain Hub

In [8]:
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [9]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

# Create ReAct Agent

In [20]:
agent  = create_react_agent(
    llm = llm,
    tools = [get_weather,get_conversion_rate,search_tool],
    prompt = prompt
)

In [21]:
agent_executor = AgentExecutor(
    agent = agent,
    tools = [get_weather,get_conversion_rate,search_tool],
    verbose = True,
    handle_parsing_errors=True
)

In [12]:
agent_executor.invoke({"input":"Plan a trip to Milan from India"})



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: Schengen visa requirements for Indian citizens to ItalyJun 16, 2026 · India is not among the visa-exempt countries for Italy or the Schengen Area. There is no visa-on-arrival or eVisa system for Indian passport holders traveling to Italy. May 9, 2026 · Indian passport holders require a Schengen visa to enter Italy, and the Italian embassy or consulate processes this as a short-stay Type C visa valid for stays up to 90 days within any 180-day period. Dec 26, 2025 · Complete Italy Schengen visa requirements, fees, processing times, and approval tips for Indian citizens. Updated December 2025. Jun 16, 2026 · India is not part of the Schengen visa-free agreement, so Indian passport holders must obtain a Schengen Type C visa before traveling to Italy. The visa is applied for through the Italian embassy or VFS Global visa application centre in the applicant's city of residence in India. Aug 11, 2025 · Want to vis

{'input': 'Plan a trip to Milan from India',
 'output': "Here's a plan for your trip to Milan from India:\n\n**1. Visa Requirements:**\n*   **Schengen Visa:** Indian passport holders **require a Schengen Type C visa** to enter Italy. This visa allows stays up to 90 days within any 180-day period.\n*   **Application Process:** You must apply for the visa through the Italian embassy, consulate, or VFS Global visa application centre in your city of residence in India. There is no visa-on-arrival or eVisa system for Indian citizens.\n\n**2. Flights:**\n*   **Direct Flights:** Air India offers direct flights from India to Milan on most days of the week (Monday, Wednesday, Thursday, Friday, Saturday, Sunday).\n*   **Airports in Milan:** Milan has two main airports; Milan Linate (LIN) is one option, and you can compare deals to find the cheapest flights.\n*   **Booking:** It's advisable to book your flights in advance, especially to find the best fares.\n\n**3. Accommodation:**\n*   **Popular

In [25]:

# ----------------------------
# System Prompt
# ----------------------------
messages = [
    SystemMessage(
        content="""
# AI Travel Agent System Prompt

You are an expert AI Travel Agent dedicated to helping users plan safe, personalized, and memorable trips. Your objective is to provide accurate, practical, and well-organized travel assistance tailored to each user's preferences, budget, and travel style.

## Core Responsibilities

Assist users with:

* Destination recommendations
* Complete trip planning
* Multi-city itineraries
* Budget planning
* Flights and transportation guidance
* Hotels and accommodation suggestions
* Restaurants and local cuisine
* Attractions and activities
* Packing recommendations
* Weather-based planning
* Visa and entry guidance
* Local transportation
* Currency and budgeting
* Travel safety
* Cultural etiquette
* Family, luxury, business, solo, adventure, and accessible travel

## Personalization

Before planning a trip, gather only the information necessary, such as:

* Departure location
* Destination
* Travel dates
* Number of travelers
* Budget
* Interests
* Preferred travel style
* Accommodation preferences
* Accessibility or dietary requirements (if relevant)

Avoid unnecessary questions.

## Tool Usage

Use available tools whenever they can provide more accurate, current, or personalized information than your internal knowledge.

Examples include:

* Checking weather before giving packing advice or planning outdoor activities.
* Converting currencies instead of estimating exchange rates.
* Searching for current information such as visa requirements, attraction hours, transportation schedules, travel advisories, local events, or other time-sensitive details.

Prefer verified tool results over assumptions. If a tool is unavailable or fails, clearly state the limitation instead of guessing.


## Planning Guidelines

When creating itineraries:

* Optimize for the user's priorities.
* Balance sightseeing with rest.
* Group nearby attractions together.
* Avoid unrealistic schedules.
* Minimize unnecessary travel.
* Suggest alternatives when appropriate.
* Explain the reasoning behind important recommendations.

## Budgeting

When relevant, provide estimated costs for:

* Transportation
* Accommodation
* Food
* Activities
* Miscellaneous expenses

Use currency conversion tools whenever needed.

## Safety

When appropriate, include:

* Safety considerations
* Local customs and etiquette
* Common tourist scams
* Emergency preparedness
* Weather-related precautions

## Response Style

Be:

* Friendly
* Professional
* Honest
* Organized
* Concise by default
* Detailed when requested

Use headings, bullet points, and tables where they improve readability.

## Accuracy

Do not fabricate prices, schedules, availability, visa rules, weather, or opening hours.

Distinguish between general travel knowledge and verified current information.

When information depends on real-time data, use available tools.

## Goal

Help users make confident travel decisions by providing accurate, personalized, and actionable travel advice while using available tools whenever they improve the quality or reliability of the response.

"""
    )
]

print("=" * 50)
print("AI Travel Planner")
print("Type 'exit' or 'quit' to end the conversation.")
print("=" * 50)

MAX_HISTORY = 10  # Number of recent messages to keep (excluding system prompt)

while True:
    user_input = input("\nYou: ").strip()

    if user_input.lower() in ["exit", "quit"]:
        print("\nSafe travels! Goodbye.")
        break

    # Add user message
    messages.append(HumanMessage(content=user_input))

    # Keep system prompt + recent conversation
    messages_to_send = [messages[0]] + messages[-MAX_HISTORY:]

    try:
        response = agent_executor.invoke({"input": messages_to_send})

        assistant_message = response["output"]

        print(f"\nTravel Planner:\n{assistant_message}")

        # Save assistant response
        messages.append(assistant_message)

    except Exception as e:
        print(f"\nError: {e}")

AI Travel Planner
Type 'exit' or 'quit' to end the conversation.



You:  Plan a trip to Milan from Delhi




> Entering new AgentExecutor chain...
The user wants to plan a trip to Milan from Delhi. To start, I need to gather more information about the user's preferences for this trip. I will ask about their travel dates, budget, and interests so I can personalize the plan.
I need to ask the user for more information to plan the trip. I will ask about their travel dates, budget, and interests.
Final Answer: I can help you plan a trip to Milan from Delhi! To get started, could you please tell me your preferred travel dates, your budget for this trip, and what your interests are (e.g., fashion, art, food, history, shopping)? This will help me create a personalized itinerary for you.

> Finished chain.

Travel Planner:
I can help you plan a trip to Milan from Delhi! To get started, could you please tell me your preferred travel dates, your budget for this trip, and what your interests are (e.g., fashion, art, food, history, shopping)? This will help me create a personalized itinerary for you.



You:  3rd July to 7th July 2026, budget friendly, all interests, solo.




> Entering new AgentExecutor chain...
The user wants to plan a budget-friendly solo trip to Milan from Delhi from July 3rd to July 7th, 2026, with all interests. I need to gather more information to provide a comprehensive plan. I should start by checking the weather in Milan for early July to help with packing recommendations and activity planning. Then, I can look into flight and accommodation options, keeping the budget in mind. I also need to consider transportation within Milan and suggest activities based on the user's broad interests.

First, I will check the weather in Milan for July.
Action: get_weather
Action Input: Milan{'location': {'city': 'Milan', 'country': 'Italy', 'local_time': '2026-07-01 13:17'}, 'current': {'temperature_c': 29.2, 'feels_like_c': 28.5, 'condition': 'Partly cloudy', 'humidity': 58, 'wind_kph': 9.7, 'uv': 8.0, 'is_day': True}, 'forecast': [{'date': '2026-07-01', 'condition': 'Sunny', 'max_temp_c': 34.2, 'min_temp_c': 21.6, 'avg_temp_c': 26.5, 'chance


You:  exit



Safe travels! Goodbye.


In [22]:
response = agent_executor.invoke({"input":"Plan a trip to Milan from India"})



> Entering new AgentExecutor chain...
I need to plan a trip to Milan from India. This involves several steps:
1.  **Flights:** I need to find flight information from India to Milan. Since I don't have a tool for flight booking, I'll use `duckduckgo_search` to look for general information on flights.
2.  **Accommodation:** I'll use `duckduckgo_search` to find information on hotels or places to stay in Milan.
3.  **Things to do in Milan:** I'll use `duckduckgo_search` to find popular tourist attractions and activities in Milan.
4.  **Weather in Milan:** I'll use the `get_weather` tool to check the current weather and forecast for Milan. This will help in packing and planning activities.
5.  **Currency Conversion:** I'll need to know the currency in Italy (Euros) and potentially the exchange rate from Indian Rupees (INR) to Euros (EUR) to help with budgeting. I'll use the `get_conversion_rate` tool for this.

Let's start with finding flights. I will search for "flights from India to Mil